In [ ]:
#installing needed libraries and models
%pip install insightface
%pip install onnxruntime-gpu
%pip install tqdm

In [ ]:
# importing libraries
import os
import cv2
import numpy as np
from tqdm import tqdm
from insightface.app import FaceAnalysis

In [ ]:
#loading arcface
app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider"]
)
app.prepare(
    ctx_id=0,
    det_size=(640,640)
)
print("ArcFace loaded.")

In [ ]:
# root path
ROOT = "dataset_split"
SPLITS = [
    "train",
    "test"
]

In [ ]:
#creating output folders for each of the train and test split
for split in SPLITS:
    os.makedirs(
        os.path.join(
            ROOT,
            split,
            "gallery_embeddings"
        ),
        exist_ok=True
    )
    os.makedirs(
        os.path.join(
            ROOT,
            split,
            "degraded_probe_embeddings"
        ),
        exist_ok=True
    )

print("Embedding folders created.")

In [ ]:
# extracting the embeddings using arcface
def get_embedding(image_path):
    img = cv2.imread(
        image_path
    )
    if img is None:
        return None
    faces = app.get(
        img
    )
    if len(faces) == 0:
        return None
    
    embedding = faces[0].embedding
    embedding = (
        embedding
        /
        np.linalg.norm(
            embedding
        )
    )

    return embedding

In [ ]:
# generating gallery embeddings
for split in SPLITS:
    input_root = os.path.join(
        ROOT,
        split,
        "gallery_cropped"
    )
    output_root = os.path.join(
        ROOT,
        split,
        "gallery_embeddings"
    )
    identities = os.listdir(
        input_root
    )
    print(
        f"\nProcessing {split}/gallery_cropped"
    )
    total_saved = 0
    total_failed = 0

    for identity in tqdm(
        identities
    ):
        src_dir = os.path.join(
            input_root,
            identity
        )
        dst_dir = os.path.join(
            output_root,
            identity
        )
        os.makedirs(
            dst_dir,
            exist_ok=True
        )
        for image_name in os.listdir(
            src_dir
        ):

            image_path = os.path.join(
                src_dir,
                image_name
            )
            emb = get_embedding(
                image_path
            )
            if emb is None:

                total_failed += 1
                continue
            save_name = (
                os.path.splitext(
                    image_name
                )[0]
                +
                ".npy"
            )
            save_path = os.path.join(
                dst_dir,
                save_name
            )
            np.save(
                save_path,
                emb
            )
            total_saved += 1

    print(
        f"\nSaved: {total_saved}"
    )
    print(
        f"Failed: {total_failed}"
    )

In [ ]:
# generating degraded probe embeddings
for split in SPLITS:
    input_root = os.path.join(
        ROOT,
        split,
        "degraded_probes"
    )
    output_root = os.path.join(
        ROOT,
        split,
        "degraded_probe_embeddings"
    )
    identities = os.listdir(
        input_root
    )
    print(
        f"\nProcessing {split}/degraded_probes"
    )
    total_saved = 0
    total_failed = 0
    for identity in tqdm(
        identities
    ):
        src_dir = os.path.join(
            input_root,
            identity
        )
        dst_dir = os.path.join(
            output_root,
            identity
        )
        os.makedirs(
            dst_dir,
            exist_ok=True
        )
        for image_name in os.listdir(
            src_dir
        ):
            image_path = os.path.join(
                src_dir,
                image_name
            )
            emb = get_embedding(
                image_path
            )
            if emb is None:

                total_failed += 1
                continue
            save_name = (
                os.path.splitext(
                    image_name
                )[0]
                +
                ".npy"
            )
            save_path = os.path.join(
                dst_dir,
                save_name
            )
            np.save(
                save_path,
                emb
            )
            total_saved += 1

    print(
        f"\nSaved: {total_saved}"
    )

    print(
        f"Failed: {total_failed}"
    )

In [ ]:
# verifying the count of embeddings genarated
for split in SPLITS:
    print(
        f"\n===== {split.upper()} ====="
    )
    for subset in [
        "gallery_embeddings",
        "degraded_probe_embeddings"
    ]:
        total = 0
        root = os.path.join(
            ROOT,
            split,
            subset
        )
        for identity in os.listdir(
            root
        ):
            total += len(
                os.listdir(
                    os.path.join(
                        root,
                        identity
                    )
                )
            )
        print(
            subset,
            "->",
            total
        )

In [ ]:
# checking one embedding 
sample_root = os.path.join(
    ROOT,
    "train",
    "gallery_embeddings"
)
person = os.listdir(
    sample_root
)[0]
file = os.listdir(
    os.path.join(
        sample_root,
        person
    )
)[0]
emb = np.load(
    os.path.join(
        sample_root,
        person,
        file
    )
)
print(
    "Embedding shape:",
    emb.shape
)
print(
    "Norm:",
    np.linalg.norm(
        emb
    )
)

In [ ]:
# creating backup zips
import shutil
for split in SPLITS:
    gallery_folder = os.path.join(
        ROOT,
        split,
        "gallery_embeddings"
    )
    degraded_folder = os.path.join(
        ROOT,
        split,
        "degraded_probe_embeddings"
    )
    shutil.make_archive(
        os.path.join(
            ROOT,
            split,
            "gallery_embeddings"
        ),
        "zip",
        gallery_folder
    )
    shutil.make_archive(
        os.path.join(
            ROOT,
            split,
            "degraded_probe_embeddings"
        ),
        "zip",
        degraded_folder
    )
print("ZIP files created.")